<a href="https://colab.research.google.com/github/dapnammoew/NHOM10_KHAIPHADULIEU/blob/main/Khai_ph%C3%A1_d%E1%BB%AF_li%E1%BB%87u.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import pandas as pd
from mlxtend.preprocessing import TransactionEncoder
from mlxtend.frequent_patterns import fpgrowth, association_rules
import ast
from google.colab import drive

print("=== BẮT ĐẦU CHẠY HỆ THỐNG KHAI PHÁ DỮ LIỆU ===")

print("1. yêu cầu quyền kết nối với Google Drive")
drive.mount('/content/drive')

print("2. đọc dữ liệu từ tệp RAW_recipes.csv")
file_path = '/content/drive/MyDrive/NHOM10_KPDL/RAW_recipes.csv'
df = pd.read_csv(file_path)

print("3 xử lý làm sạch định dạng mảng nguyên liệu")
df['ingredients'] = df['ingredients'].apply(ast.literal_eval)

print("4. Đang kết xuất và lưu từ điển 230k công thức vào Drive")
df_recipes = df[['name', 'ingredients']]
df_recipes.to_csv('/content/drive/MyDrive/NHOM10_KPDL/clean_recipes.csv', index=False)
print("   -> HOÀN TẤT: Đã lưu thành công file 'clean_recipes.csv' !")

print("5. trích xuất ngẫu nhiên 50,000 công thức để chạy thuật toán")
df_sample = df.sample(n=50000, random_state=42)
transactions = df_sample['ingredients'].tolist()

print("6. chuyển đổi cấu trúc sang Ma trận thưa")
te = TransactionEncoder()
te_ary = te.fit(transactions).transform(transactions, sparse=True)
df_trans = pd.DataFrame.sparse.from_spmatrix(te_ary, columns=te.columns_)

print("7. huấn luyện thuật toán FP-Growth (min_support = 0.005)")
frequent_itemsets = fpgrowth(df_trans, min_support=0.005, use_colnames=True)

print("8.  khai phá các luật kết hợp phổ biến")
rules = association_rules(frequent_itemsets, metric="lift", min_threshold=1.0)
rules['antecedents'] = rules['antecedents'].apply(lambda x: list(x))
rules['consequents'] = rules['consequents'].apply(lambda x: list(x))

print("9. lưu tập Luật kết hợp vào Google Drive...")
rules = rules[['antecedents', 'consequents', 'support', 'confidence', 'lift']]
rules.to_csv('/content/drive/MyDrive/NHOM10_KPDL/ingredient_rules.csv', index=False)
print("Đã lưu thành công file 'ingredient_rules.csv'!")